# 🗡️ Hey Aragorn Wake Word Training

This notebook trains a custom wake word model for "Hey Aragorn" using micro-wake-word.

**Run each cell in order.** The notebook will guide you through each step.

## Step 1: Setup Environment

Install all required dependencies. This takes ~3-5 minutes.

In [ ]:
import subprocess
import sys
import os

print("📦 Installing dependencies...")

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "tensorflow==2.16.1", "numpy==1.26.4", "pyyaml", "scipy", "datasets", "mmap-ninja", "tqdm", "audiomentations", "webrtcvad-wheels"], check=True)

print("✅ Dependencies installed!")

## Step 2: Clone Required Repositories

In [ ]:
import os

print("📥 Cloning repositories...")

if not os.path.exists("microWakeWord"):
    subprocess.run(["git", "clone", "https://github.com/kahrendt/microWakeWord.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", "microWakeWord"], check=True)

if not os.path.exists("piper-sample-generator"):
    subprocess.run(["git", "clone", "https://github.com/rhasspy/piper-sample-generator.git"], check=True)

print("✅ Repositories ready!")

## Step 3: Download Piper Voice Model

This is the TTS model used to generate "Hey Aragorn" samples. (~200MB)

In [ ]:
import urllib.request

os.makedirs("piper-sample-generator/models", exist_ok=True)

model_url = "https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt"
model_path = "piper-sample-generator/models/en_US-libritts_r-medium.pt"

if not os.path.exists(model_path):
    print("⬇️ Downloading Piper voice model...")
    urllib.request.urlretrieve(model_url, model_path)
    print("✅ Voice model downloaded!")
else:
    print("✅ Voice model already exists")

## Step 4: Configure Wake Word

**Edit the settings below:**

In [ ]:
# ⚙️ CONFIGURATION - Edit these values

TARGET_WORD = "hey_air_uh_gorn"  # Phonetic spelling for "Hey Aragorn"
NUM_SAMPLES = 1000                # Number of TTS samples to generate
TRAINING_STEPS = 10000            # Training steps (more = better accuracy)

print(f"🎯 Target word: {TARGET_WORD}")
print(f"🔢 Samples to generate: {NUM_SAMPLES}")
print(f"🔄 Training steps: {TRAINING_STEPS}")

## Step 5: Generate Wake Word Samples

This generates TTS audio samples of "Hey Aragorn". Takes ~5-10 minutes.

In [ ]:
print(f"🎙️ Generating {NUM_SAMPLES} samples for '{TARGET_WORD}'...")
print("⏳ This will take several minutes...")

result = subprocess.run([
    sys.executable,
    "piper-sample-generator/generate_samples.py",
    TARGET_WORD,
    "--max-samples", str(NUM_SAMPLES),
    "--batch-size", "100",
    "--model", "piper-sample-generator/models/en_US-libritts_r-medium.pt",
    "--output-dir", "generated_samples",
], capture_output=True, text=True)

if result.returncode == 0:
    sample_count = len([f for f in os.listdir("generated_samples") if f.endswith('.wav')])
    print(f"✅ Generated {sample_count} samples!")
else:
    print("❌ Error generating samples:")
    print(result.stderr)

## Step 6: Download Augmentation Data

Download room impulse responses and background noise for data augmentation.

In [ ]:
print("📥 Downloading augmentation data...")

os.makedirs("mit_rirs", exist_ok=True)
if not os.listdir("mit_rirs"):
    print("  Downloading MIT RIRs...")
    subprocess.run(["wget", "-q", "-O", "/tmp/ir_mit.zip", "https://mcdermottlab.mit.edu/Reverb/IR_MIT_Survey.zip"], check=True)
    subprocess.run(["unzip", "-q", "/tmp/ir_mit.zip", "-d", "mit_rirs"], check=True)

os.makedirs("audioset_16k", exist_ok=True)
os.makedirs("fma_16k", exist_ok=True)

print("✅ Augmentation data ready!")

## Step 7: Generate Spectrograms

Convert audio samples to spectrogram features for training.

In [ ]:
print("📊 Generating spectrograms...")

sys.path.insert(0, 'microWakeWord')

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
from mmap_ninja.ragged import RaggedMmap

clips = Clips(
    input_directory="generated_samples",
    file_pattern='*.wav',
    max_clip_duration_s=None,
    remove_silence=False,
    random_split_seed=10,
    split_count=0.1,
)

augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        "SevenBandParametricEQ": 0.1,
        "TanhDistortion": 0.1,
        "PitchShift": 0.1,
        "BandStopFilter": 0.1,
        "AddColorNoise": 0.1,
        "AddBackgroundNoise": 0.75,
        "Gain": 1.0,
        "RIR": 0.5,
    },
    impulse_paths=["mit_rirs"],
    background_paths=["fma_16k", "audioset_16k"],
    background_min_snr_db=-5,
    background_max_snr_db=10,
    min_jitter_s=0.195,
    max_jitter_s=0.205,
)

os.makedirs("generated_augmented_features/training", exist_ok=True)

spectrograms = SpectrogramGeneration(
    clips=clips,
    augmenter=augmenter,
    slide_frames=10,
    step_ms=10,
)

RaggedMmap.from_generator(
    out_dir="generated_augmented_features/training/wakeword_mmap",
    sample_generator=spectrograms.spectrogram_generator(split="train", repeat=2),
    batch_size=100,
    verbose=True,
)

print("✅ Spectrograms generated!")

## Step 8: Download Negative Datasets

Download pre-generated negative samples (speech, noise, etc.)

In [ ]:
print("📥 Downloading negative datasets...")

os.makedirs("negative_datasets", exist_ok=True)

base_url = "https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/"
files = ['dinner_party.zip', 'dinner_party_eval.zip', 'no_speech.zip', 'speech.zip']

for fname in files:
    zip_path = f"negative_datasets/{fname}"
    if not os.path.exists(zip_path.replace('.zip', '')):
        print(f"  Downloading {fname}...")
        urllib.request.urlretrieve(base_url + fname, zip_path)
        subprocess.run(["unzip", "-q", zip_path, "-d", "negative_datasets"], check=True)
        os.remove(zip_path)

print("✅ Negative datasets ready!")

## Step 9: Create Training Configuration

In [ ]:
import yaml

config = {
    "window_step_ms": 10,
    "train_dir": "trained_models/wakeword",
    "spectrogram_length": 204,
    "stride": 3,
    "features": [
        {
            "features_dir": "generated_augmented_features",
            "sampling_weight": 2.0,
            "penalty_weight": 1.0,
            "truth": True,
            "truncation_strategy": "truncate_start",
            "type": "mmap",
        },
        {
            "features_dir": "negative_datasets/speech",
            "sampling_weight": 10.0,
            "penalty_weight": 1.0,
            "truth": False,
            "truncation_strategy": "random",
            "type": "mmap",
        },
        {
            "features_dir": "negative_datasets/dinner_party",
            "sampling_weight": 10.0,
            "penalty_weight": 1.0,
            "truth": False,
            "truncation_strategy": "random",
            "type": "mmap",
        },
        {
            "features_dir": "negative_datasets/no_speech",
            "sampling_weight": 5.0,
            "penalty_weight": 1.0,
            "truth": False,
            "truncation_strategy": "random",
            "type": "mmap",
        },
        {
            "features_dir": "negative_datasets/dinner_party_eval",
            "sampling_weight": 0.0,
            "penalty_weight": 1.0,
            "truth": False,
            "truncation_strategy": "split",
            "type": "mmap",
        },
    ],
    "training_steps": [TRAINING_STEPS],
    "positive_class_weight": [1],
    "negative_class_weight": [20],
    "learning_rates": [0.001],
    "batch_size": 128,
    "time_mask_max_size": [0],
    "time_mask_count": [0],
    "freq_mask_max_size": [0],
    "freq_mask_count": [0],
    "eval_step_interval": 500,
    "clip_duration_ms": 1500,
    "target_minimization": 0.9,
    "minimization_metric": None,
    "maximization_metric": "average_viable_recall",
}

with open("training_parameters.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print("✅ Configuration saved to training_parameters.yaml")

## Step 10: Train Model

**This is the big step!** Training takes 1-3 hours depending on the number of steps.

The model will be saved to `trained_models/wakeword/`

In [ ]:
print("🚀 Starting training...")
print(f"⏳ This will take approximately {TRAINING_STEPS // 10000} hour(s)...")
print("☕ Go grab a coffee!")

result = subprocess.run([
    sys.executable, "-m", "microwakeword.model_train_eval",
    "--training_config=training_parameters.yaml",
    "--train=1",
    "--restore_checkpoint", "1",
    "--test_tf_nonstreaming", "0",
    "--test_tflite_nonstreaming", "0",
    "--test_tflite_nonstreaming_quantized", "0",
    "--test_tflite_streaming", "0",
    "--test_tflite_streaming_quantized", "1",
    "mixednet",
    "--pointwise_filters", "64,64,64,64",
    "--repeat_in_block", "1, 1, 1, 1",
    "--mixconv_kernel_sizes", "'[5], [7,11], [9,15], [23]'",
    "--residual_connection", "0,0,0,0",
    "--first_conv_filters", "32",
    "--first_conv_kernel_size", "5",
    "--stride", "3",
])

if result.returncode == 0:
    print("✅ Training complete!")
else:
    print("❌ Training failed")
    if result.stderr:
        print(result.stderr)

## Step 11: Verify Output

Check that the model was created successfully.

In [ ]:
model_path = "trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"

if os.path.exists(model_path):
    size_kb = os.path.getsize(model_path) / 1024
    print(f"✅ Model created successfully!")
    print(f"📁 Location: {model_path}")
    print(f"📦 Size: {size_kb:.1f} KB")
    print(f"\n🎉 Your 'Hey Aragorn' wake word model is ready!")
else:
    print("❌ Model not found at expected location")
    print("Checking trained_models/...")
    for root, dirs, files in os.walk("trained_models"):
        for f in files:
            if f.endswith('.tflite'):
                print(f"  Found: {os.path.join(root, f)}")

## Next Steps

1. **Download the model** from the file explorer on the left
2. **Create a model manifest** (JSON file with metadata)
3. **Flash to your Satellite1** using ESPHome

The model file is: `trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite`